# College Mayor Elections – Developer Walkthrough

This notebook helps you run the system locally (API + DB + Web), test it, and try a few API calls.

**Dev users (seeded on first run)**
- Admin: `admin@example.com` / `admin123`
- Voter: `voter@example.com` / `voter123`


## 1) Start everything with Docker
From the repo root:

```bash
docker compose up --build
```

- Web: http://localhost:5173
- API: http://localhost:8000/docs
- DB: Postgres on localhost:5432


In [ ]:
import requests
API = 'http://localhost:8000'

def login(email, password):
    r = requests.post(API + '/auth/login', json={'email': email, 'password': password})
    r.raise_for_status()
    return r.json()['access_token']

admin_token = login('admin@example.com', 'admin123')
voter_token = login('voter@example.com', 'voter123')
admin_token[:20], voter_token[:20]


In [ ]:
import datetime
now = datetime.datetime.now(datetime.timezone.utc)
payload = {
  'name': 'Election 2026',
  'starts_at': now.isoformat(),
  'ends_at': (now + datetime.timedelta(days=1)).isoformat()
}
headers_admin = {'Authorization': f'Bearer {admin_token}'}
r = requests.post(API + '/elections', json=payload, headers=headers_admin)
r.status_code, r.json()


In [ ]:
election_id = r.json()['id']
r1 = requests.post(API + f'/elections/{election_id}/candidates', json={'full_name':'Alice', 'manifesto':'More buses'}, headers=headers_admin)
r2 = requests.post(API + f'/elections/{election_id}/candidates', json={'full_name':'Bob', 'manifesto':'More scholarships'}, headers=headers_admin)
r1.status_code, r2.status_code


In [ ]:
requests.post(API + f'/elections/{election_id}/open', headers=headers_admin).json()


In [ ]:
candidate_id = r1.json()['id']
headers_voter = {'Authorization': f'Bearer {voter_token}'}
vote = requests.post(API + f'/elections/{election_id}/vote', json={'candidate_id': candidate_id}, headers=headers_voter)
vote.status_code, vote.text


In [ ]:
results = requests.get(API + f'/elections/{election_id}/results', headers=headers_voter)
results.status_code, results.json()


## 2) Run tests
Backend:
```bash
cd backend
pytest -q
```
Frontend:
```bash
cd frontend
npm install
npm test
```
